# Week 34, Day 4: Correlated variability: measuring and interpreting noise correlations

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w034/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Correlation depends on the window

Two neurons fire Poisson spikes at 20 Hz, each multiplied by exp(0.4 s(t)), where s(t) is a shared Ornstein-Uhlenbeck process with a 50 ms time constant. For 300 one-second trials, count spikes in non-overlapping windows of 10, 50, 200 and 1,000 ms, and compute the correlation of the two neurons' counts across all windows, `rsc[w]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2011)
trials, T = 300, 1000
a = np.exp(-1 / 50.0); x = np.zeros(trials); s = np.empty((trials, T)); xi = r.normal(size=(trials, T))
for k in range(T):                                  # shared fluctuation, one value per trial per ms
    x = a * x + np.sqrt(1 - a * a) * xi[:, k]; s[:, k] = x
rate = 20 * np.exp(0.4 * s - 0.08) / 1000           # spikes per ms, mean about 20 Hz
sp1 = r.uniform(size=(trials, T)) < rate
sp2 = r.uniform(size=(trials, T)) < rate

rsc = {}
for w in (10, 50, 200, 1000):
    n1 = sp1.reshape(trials, -1, w).sum(2).ravel()
    n2 = sp2.reshape(trials, -1, w).sum(2).ravel()
    rsc[w] = ...
    print(f"window {w:4d} ms: r_sc = {rsc[w]:.3f}")

In [ ]:
import numpy as np
def test_corr():
    "rsc holds the count correlation for each window, rising from under 0.08 at 10 ms to over 0.12 at 200 ms"
    assert rsc[10] < 0.08 and rsc[200] > 0.12 and rsc[1000] > 3 * rsc[10]

test_corr()
print("All checks passed")

### Exercise: Diminishing returns

Simulate N correlated neurons with unit variance and pairwise correlation 0.1 by giving each a shared Gaussian term (variance 0.1) plus a private one (variance 0.9), over 5,000 trials. Compute the variance of their average for N = 1, 10, 100 and 1,000 (`sim[N]`) and compare with `theory(N)`.

In [ ]:
rho = 0.1

def theory(N):
    return ...

Ns = (1, 10, 100, 1000)
sim = {}
for N in Ns:
    shared = r.normal(0, np.sqrt(rho), (5000, 1))
    private = r.normal(0, np.sqrt(1 - rho), (5000, N))
    sim[N] = (shared + private).mean(1).var()
    print(f"N = {N:4d}: variance of the average {sim[N]:.4f}, formula {theory(N):.4f}, independent neurons {1 / N:.4f}")

In [ ]:
import numpy as np
def test_formula():
    "the simulated variance of the average matches the formula within 10%"
    assert all(abs(sim[N] / theory(N) - 1) < 0.1 for N in Ns)

def test_floor():
    "with rho = 0.1, going from 100 to 1,000 neurons reduces the variance by less than 15%"
    assert theory(1000) > 0.85 * theory(100)

test_formula()
test_floor()
print("All checks passed")

### Exercise: Same tuning, opposite tuning

Write `fisher_info(fp, rho)` for two neurons with unit noise variance and correlation rho. Compute it for signal changes f' = (1, 1) ('same') and (1, -1) ('opposite') at rho = -0.3, 0 and 0.3.

In [ ]:
def fisher_info(fp, rho):
    Sigma = np.array([[1.0, rho], [rho, 1.0]])
    return ...

info = {"same": {}, "opposite": {}}
for name, fp in (("same", np.array([1.0, 1.0])), ("opposite", np.array([1.0, -1.0]))):
    for rho_ in (-0.3, 0.0, 0.3):
        info[name][rho_] = fisher_info(fp, rho_)
    print(f"{name:8s} tuning: I = {info[name][-0.3]:.2f} (rho -0.3), {info[name][0.0]:.2f} (0), {info[name][0.3]:.2f} (0.3)")

In [ ]:
import numpy as np
def test_info():
    "fisher_info computes f' Sigma^-1 f'"
    assert abs(fisher_info(np.array([1.0, 1.0]), 0.0) - 2) < 1e-12

def test_sign():
    "with rho = 0.3, information falls for same-sign tuning and rises for opposite-sign tuning"
    assert info["same"][0.3] < info["same"][0.0] < info["opposite"][0.3]

test_info()
test_sign()
print("All checks passed")